In [1]:
from environment import Environment
from nnsight import LanguageModel
import sys
import torch as t
from getpass import getpass
import os

REPLICATE_API_TOKEN = getpass()
os.environ["REPLICATE_API_TOKEN"] = "r8_YwrSiHNmeiOLZrFbDsh0iWGLmY55lmr3iQKPQ"

In [2]:
model = LanguageModel("openai-community/gpt2", device_map='auto', dispatch=True)

In [3]:
# !git clone https://github.com/jbloomAus/mats_sae_training.git
# %cd mats_sae_training
# !pip install -r requirements.txt

sys.path.append("../mats_sae_training")

from sae_training.sparse_autoencoder import SparseAutoencoder

t.set_grad_enabled(False)

In [4]:
from huggingface_hub import hf_hub_download

REPO_ID = "jbloom/GPT2-Small-SAEs"

sae_list = []
n_layers = 12

local_dir = "../jbloom_dictionaries"

for layer in range(n_layers):
    filename =  f"final_sparse_autoencoder_gpt2-small_blocks.{layer}.hook_resid_pre_24576.pt"
    resid_sae = hf_hub_download(repo_id = REPO_ID, filename = filename, local_dir=local_dir)

    save_path = f"{local_dir}/{filename}"
    sae = SparseAutoencoder.load_from_pretrained(save_path)
    sae.to("cuda:0")
    
    sae_list.append(sae)

print("Loaded dictionaries")

Loaded dictionaries


In [5]:
env = Environment(
    model=model, 
    sae_list=sae_list,
)

In [6]:
env.load(
    layer=10,
    feature_id=7000,
)

100%|██████████| 13/13 [00:38<00:00,  2.93s/it]

Activation Cache Size: torch.Size([1950, 128, 24576])


In [7]:
mixed_examples_list = env.d_scorer.get_mixed_examples_list(7000)

AssertionError: 

In [7]:
explanation_list = env.explainer.explain(7000)

  0%|          | 0/2 [00:00<?, ?it/s]

  0%|          | 0/2 [00:00<?, ?it/s]


ReplicateError: ReplicateError Details:
title: Unauthenticated
status: 401
detail: You did not pass a valid authentication token

In [ ]:
class DetectionScorer:
  def __init__(self, cfg):
    self.cfg = cfg


  def get_mixed_examples_list(self, feature_id):
    n_real, n_fake = self.cfg.n_real, self.cfg.n_real
    batch_size = 2*n_real
    n_batches = self.cfg.n_batches
    real_ids = self.cfg.real_ids
    left_context = self.cfg.left_context
    right_context = self.cfg.right_context

    # should change to use different examples from those the explainer saw
    assert (len(real_ids) == n_real)
    top_acts, top_inds = topk(FEATURE_ACTS[:,:, feature_id], n_batches*n_real)

    true_examples_list = []
    for i in range(n_batches*n_real):
      start_pos = max(0, top_inds[i][1] - left_context)
      end_pos = min(TOKS.size(1), top_inds[i][1] + right_context + 1)
      true_example_toks = TOKS[top_inds[i][0], start_pos : end_pos]
      true_examples_list.append(true_example_toks)

    mixed_examples_list = []
    while len(mixed_examples_list) < n_batches*batch_size:
      b = t.randint(0, FEATURE_ACTS.size(0), [1]).squeeze()
      s = t.randint(left_context, FEATURE_ACTS.size(1)-right_context, [1]).squeeze()

      fake_example_toks = TOKS[b, s - left_context : s + right_context + 1]
      fake_example_acts = FEATURE_ACTS[b, s - left_context : s + right_context + 1, feature_id]
      if sum(fake_example_acts) < 0.01:
        mixed_examples_list.append(fake_example_toks)

    for b in range(n_batches):
      for i in range(n_real):
        mixed_examples_list[b*batch_size + real_ids[i]] = true_examples_list[b*n_real + i]

    mixed_examples_list = model.tokenizer.batch_decode(mixed_examples_list)

    mixed_examples_list = [example.replace("{", "{{").replace("}", "}}") for example in mixed_examples_list]


    self.mixed_examples_list = mixed_examples_list

  def score(self, feature_id, explanation_list):
    self.get_mixed_examples_list(feature_id)
    score_list = []

    for explanation in tqdm(explanation_list):
      if self.cfg.verbose: print(f"EXPLANATION:     {explanation}\n\n\n")
      summed_detection_rate = 0.0
      summed_false_pos_rate = 0.0

      for b in range(self.cfg.n_batches):

        mixed_examples_list = self.mixed_examples_list[b*2*self.cfg.n_real : (b+1)*2*self.cfg.n_real]
        mixed_examples_str = ''
        for i in range(len(mixed_examples_list)):
          mixed_examples_str += f'Example {i+1}: {mixed_examples_list[i]}\n'

        if self.cfg.verbose: print(f"---------------BATCH {b}  MIXED EXAMPLES----------------\n\n\n", mixed_examples_str, "n\n\n\n\n\n\n")

        detection_rate, false_pos_rate = self.query_llm(explanation, mixed_examples_str)
        summed_detection_rate += detection_rate
        summed_false_pos_rate += false_pos_rate

      detection_rate = summed_detection_rate /  self.cfg.n_batches
      false_pos_rate = summed_false_pos_rate /  self.cfg.n_batches
      score_list.append((detection_rate, false_pos_rate))
      if self.cfg.verbose: print(f"SCORE:           {detection_rate} {false_pos_rate}\n\n\n")

    return score_list

  def query_llm(self, explanation, mixed_examples_str):
    input = {
        "prompt": mixed_examples_str,
        "prompt_template": get_dscorer_template(mixed_examples_str, explanation),
        "max_tokens" : 1000,
        "temperature" : 0.0
    }

    output = replicate.run(
        "meta/meta-llama-3-70b-instruct",
        input=input
    )


    output_str = ''
    for i in output:
      output_str += i

    if self.cfg.verbose: print("---------------DETECTION SCORER OUTPUT-------------\n\n\n", output_str, "\n\n\n\n\n\n")


    nums = self.extract_numbers_from_string(output_str)

    if self.cfg.verbose: print("output: ", nums)
    detection_rate = sum([num-1 in self.cfg.real_ids for num in nums]) / self.cfg.n_real
    false_pos_rate = sum([num-1 not in self.cfg.real_ids for num in nums]) / self.cfg.n_real

    return detection_rate, false_pos_rate

  def extract_numbers_from_string(self, s):
    # Find the starting index of the list
    start_index = s.find('[')
    # Find the ending index of the list
    end_index = s.find(']')

    if start_index == -1 or end_index == -1:
        # Return an empty list if '[' or ']' is not found in the string
        return []

    # Extract the substring containing the list of numbers
    numbers_str = s[start_index+1:end_index]
    # Split the substring by comma and convert each part to integer
    numbers = [int(num) for num in numbers_str.split(',')]

    return numbers